In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [14]:

data=pd.read_csv("loan_approval_dataset.csv")
for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].str.strip()


In [15]:
#explore data 
print(data.shape)
print(data.dtypes)
data.head()

(4269, 13)
loan_id                       int64
 no_of_dependents             int64
 education                   object
 self_employed               object
 income_annum                 int64
 loan_amount                  int64
 loan_term                    int64
 cibil_score                  int64
 residential_assets_value     int64
 commercial_assets_value      int64
 luxury_assets_value          int64
 bank_asset_value             int64
 loan_status                 object
dtype: object


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [16]:
print("missing values per column:")
print(data.isnull().sum())

print("\nclass balance:")
print(data[" loan_status"].value_counts())
print(data[" loan_status"].value_counts(normalize=True).round(3))# normalize (entre 0 et 1), round 3 (apres la virgule)

missing values per column:
loan_id                      0
 no_of_dependents            0
 education                   0
 self_employed               0
 income_annum                0
 loan_amount                 0
 loan_term                   0
 cibil_score                 0
 residential_assets_value    0
 commercial_assets_value     0
 luxury_assets_value         0
 bank_asset_value            0
 loan_status                 0
dtype: int64

class balance:
 loan_status
Approved    2656
Rejected    1613
Name: count, dtype: int64
 loan_status
Approved    0.622
Rejected    0.378
Name: proportion, dtype: float64


In [17]:
#the target column
y=data[' loan_status']
x=data.drop(' loan_status',axis=1)

In [18]:
#identify the numerical and categorical columns
numerical_cols = x.select_dtypes(include="number").columns.tolist()
categorical_cols = x.select_dtypes(include="object").columns.tolist()
print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['loan_id', ' no_of_dependents', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value']
Categorical columns: [' education', ' self_employed']


In [19]:
# split the dataset
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print("Train size:", x_train.shape, " Test size:", x_test.shape)

Train size: (3415, 12)  Test size: (854, 12)


In [20]:
#data pre_processing
# first of all we have to fill the NAN for numerical cols
numeric_transformers=Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])
#then we fill the categorical cols, then we transforme the text into number (0 OR 1)
categorical_transformer=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

#column transformer
preprocessor=ColumnTransformer(
    transformers=[
    ("num",numeric_transformers,numerical_cols,),
    ("cat",categorical_transformer,categorical_cols)

    ]
)



In [23]:
models = {
    "logistic Regression": LogisticRegression(random_state=42,max_iter=5000),
    "Random Forest":RandomForestClassifier(random_state=42,n_estimators=300),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

fitted_pipelines = {}
results = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", model)])
    pipe.fit(x_train, y_train)
    fitted_pipelines[name] = pipe

    preds = pipe.predict(x_test)
    acc = accuracy_score(y_test, preds)
    cv_scores = cross_val_score(pipe, x_train, y_train, cv=5, scoring="accuracy")
    results[name] = {"test_accuracy": acc, "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std()}

    print(f"\n{'='*60}")
    print(f"{name}")
    print('='*60)
    print(f"Test accuracy:        {acc:.4f}")
    print(f"5-fold CV accuracy:   {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, preds))
    print("Confusion matrix (rows=actual, cols=predicted):")
    print(pd.DataFrame(
        confusion_matrix(y_test, preds),
        index=[f"actual_{c}" for c in pipe.classes_],
        columns=[f"pred_{c}" for c in pipe.classes_]
    ))


logistic Regression
Test accuracy:        0.8220
5-fold CV accuracy:   0.8056 +/- 0.0170

Classification report:
              precision    recall  f1-score   support

    Approved       0.82      0.92      0.87       536
    Rejected       0.83      0.66      0.73       318

    accuracy                           0.82       854
   macro avg       0.82      0.79      0.80       854
weighted avg       0.82      0.82      0.82       854

Confusion matrix (rows=actual, cols=predicted):
                 pred_Approved  pred_Rejected
actual_Approved            492             44
actual_Rejected            108            210

Random Forest
Test accuracy:        0.9824
5-fold CV accuracy:   0.9769 +/- 0.0033

Classification report:
              precision    recall  f1-score   support

    Approved       0.98      0.99      0.99       536
    Rejected       0.98      0.97      0.98       318

    accuracy                           0.98       854
   macro avg       0.98      0.98      0.98    

In [24]:
summary = pd.DataFrame(results).T.sort_values("test_accuracy", ascending=False)
summary

,test_accuracy,cv_mean,cv_std
Random Forest,0.982436,0.976867,0.003261
Gradient Boosting,0.975410,0.977452,0.004685
logistic Regression,0.822014,0.805564,0.017009
